In [14]:
import json
import requests
import re
import sys
from pathlib import Path

In [15]:
def create_prompt(context_chunk, category):
    return f"""<|im_start|>system
Bạn là chuyên gia học vụ đại học và trợ lý pháp lý giáo dục. 
QUY TẮC CỐT LÕI:
1. Chỉ sử dụng duy nhất thông tin trong văn bản được cung cấp.
2. Tuyệt đối không suy diễn hoặc dùng kiến thức bên ngoài (đặc biệt là các con số).
3. Nếu không có thông tin phù hợp, chỉ trả về duy nhất ký tự "0".<|im_end|>
<|im_start|>user
NHIỆM VỤ:
- Đọc nội dung quy chế: "{context_chunk}"
- Sinh DUY NHẤT 01 cặp CÂU HỎI – CÂU TRẢ LỜI (Q&A) cho CHỦ ĐỀ: {category}
- Nếu nội dung không liên quan đến chủ đề, lập tức trả về '0'
- Nếu nội dung không có đủ thông tin để tạo câu hỏi, lập tức trả vê '0'

YÊU CẦU VỀ CÂU HỎI (Q):
- Mang tính THỰC TẾ (Facebook, Diễn đàn sinh viên). 
- Ví dụ: "Em có được...", "Nếu... thì sao?", "Liệu có bị...".
- Không hỏi máy móc kiểu "Điều X quy định gì?".

YÊU CẦU VỀ CÂU TRẢ LỜI (A):
- Trả lời ĐÚNG và TRỰC TIẾP theo quy chế.
- Văn phong học vụ, rõ ràng, chính xác.

MỤC TIÊU: Fine-tune mô hình và làm RAG.

ĐỊNH DẠNG OUTPUT (BẮT BUỘC):
Q: <Câu hỏi>
A: <Câu trả lời>

LƯU Ý: Nếu văn bản không chứa thông tin liên quan đến các chủ đề được cung cấp hoặc không đủ dữ liệu để sinh câu hỏi, trả về duy nhất "0".<|im_end|>
<|im_start|>assistant
"""

In [ ]:
# ================= CẤU HÌNH =================
INPUT_FILE = "HNMU_Chunks_Segment.json"
OUTPUT_FILE = "dataset_qa.json"
API_URL = "http://localhost:8080/v1/chat/completions"
TEMPERATURE = 0
# ============================================

category = [        
    "Đăng ký học tập – tín chỉ – học kỳ",
    "Điều kiện học lại – cải thiện điểm – thi lại",
    "Đánh giá, chấm điểm, xếp loại học lực",
    "Cảnh báo học tập – buộc thôi học",
    "Đồ án, khóa luận, học phần thay thế",
    "Nghỉ học tạm thời – bảo lưu – thôi học",
    "Công nhận tín chỉ – miễn học – chứng chỉ ngoại ngữ & tin học",
    "Chuyển ngành, chuyển trường, học song ngành",
    "Điều kiện tốt nghiệp & xếp hạng bằng"
]

def construct_full_context(item):
    parts = []
    
    # 1. Thêm Tiêu đề chương
    if item.get("Level 1"):
        parts.append(item["Level 1"].upper())
        
    # 2. Thêm Tiêu đề điều
    if item.get("Level 2"):
        parts.append(item["Level 2"])
        
    # 3. Thêm nội dung chính
    if item.get("Article"):
        parts.append(item["Article"])
        
    # 4. Thêm các mục chi tiết
    content_list = item.get("Content", [])
    if isinstance(content_list, list):
        for line in content_list:
            parts.append(line)
    elif isinstance(content_list, str):
        parts.append(content_list)
        
    # Gộp lại bằng dấu xuống dòng
    return "\n".join(parts)


def parse_qa_text(text):
    """
    Phân tích văn bản thô (Q: ... A: ...) thành danh sách dictionary.
    Có cập nhật thêm tham số current_topic để điền tự động.
    """
    qa_list = []
    pattern = r"Q:\s*(.*?)\s*\nA:\s*(.*?)(?=\nQ:|$)"
    
    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
    
    for q, a in matches:
        q_clean = q.strip()
        a_clean = a.strip()
        
        if q_clean and a_clean:
            qa_list.append({
                "Câu hỏi": q_clean,
                "Câu trả lời": a_clean
            })
            
    return qa_list

def process_chunks(minidx, maxidx):
    if not Path(INPUT_FILE).exists():
        print(f"❌ Không tìm thấy file: {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"📂 Đã tải {len(chunks)} chunks.")
    all_qa_pairs = []

    for index, chunk in enumerate(chunks):
        if index < minidx-1: continue
        if index > maxidx-1: break

        text_content = construct_full_context(chunk)
        
        if not text_content or len(str(text_content)) < 10: continue

        print(f"\n--- Processing chunk {index + 1} ---")

        # DUYỆT TỪNG CATEGORY ĐỂ ĐẢM BẢO CHÍNH XÁC
        for cat in category:
            payload = {
                "messages": [
                    {"role": "system", "content": "Bạn là chuyên gia học vụ. Chỉ tạo sinh dữ liệu dựa trên văn bản. Nếu không đủ thông tin, trả về giá trị duy nhất '0'. KHÔNG TỰ BỊA SỐ LIỆU."},
                    {"role": "user", "content": create_prompt(text_content, cat)}
                ],
                "temperature": TEMPERATURE,
                "max_tokens": 1024,
            }

            try:
                resp = requests.post(API_URL, json=payload, timeout=30)
                if resp.status_code == 200:
                    raw_content = resp.json()['choices'][0]['message']['content'].strip()
                    
                    
                    
                    if raw_content == "0" or "A: 0" in raw_content or "A:0" in raw_content or len(raw_content) < 5:
                        print(f"\n📍[RAW]: {raw_content}")
                        print(f"📍 [SKIP] Category: {cat} ...")
                        continue
                    
                    print(f"\n📍[RAW]:\n{raw_content}")
                    print(f"📍 Category: {cat} -> Đang trích xuất...")
                    parsed_items = parse_qa_text(raw_content)
                    
                    if parsed_items:
                        for item in parsed_items:
                            item["Chủ đề"] = cat
                        
                        all_qa_pairs.extend(parsed_items)
                        
                        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                            json.dump(all_qa_pairs, f, ensure_ascii=False, indent=4)
                
            except Exception as e:
                print(f"❌ Lỗi tại category '{cat}': {e}")

    print(f"\n🎉 HOÀN TẤT! Tổng số cặp câu hỏi: {len(all_qa_pairs)}")

if __name__ == "__main__":
    process_chunks(4, 10)

📂 Đã tải 156 chunks.

--- Processing chunk 4 ---
[RAW]:
Q: Em có được đăng ký học 120 tín chỉ trong một học kỳ không, theo quy định hiện hành?
A: Theo quy chế, khối lượng kiến thức tối thiểu đối với một trình độ đào tạo của giáo dục đại học là 120 tín chỉ, nhưng không xác định rõ số tín chỉ có thể đăng ký trong một học kỳ cụ thể. Bạn nên liên hệ với bộ phận quản lý học tập của trường để có thông tin chính xác.
📍 Category: Đăng ký học tập – tín chỉ – học kỳ -> Đang trích xuất...
[RAW]:
0
📍 [SKIP] Category: Điều kiện học lại – cải thiện điểm – thi lại ...
[RAW]:
Q: Nếu tôi tích luỹ đủ 150 tín chỉ cho một ngành có thời gian đào tạo 5 năm, tôi có thể được xếp loại học lực là tốt không?
A: Theo quy chế, khi tích luỹ đủ 150 tín chỉ cho một ngành có thời gian đào tạo 5 năm, người học đáp ứng khối lượng kiến thức tối thiểu và có thể được xếp loại học lực tốt.
📍 Category: Đánh giá, chấm điểm, xếp loại học lực -> Đang trích xuất...
[RAW]:
0
📍 [SKIP] Category: Cảnh báo học tập – buộc thôi học ...